In [3]:
!pip install streamlit requests beautifulsoup4 scrapy crochet wordcloud matplotlib nltk
!npm install localtunnel

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.4/365.4 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 492.7/492.7 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.6/270.6 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.4/106.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 5.9 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
added 22 packages in 3s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸

In [8]:
# Cria a pasta utils
!mkdir -p utils

In [9]:
%%writefile utils/processamento.py
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)
palavras_vazias = set(stopwords.words('portuguese'))

def limpar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r'\W+', ' ', texto)
    todas_palavras = texto.split()

    palavras_validas = []
    for palavra in todas_palavras:
        if palavra not in palavras_vazias and len(palavra) > 2:
            palavras_validas.append(palavra)

    texto_final = " ".join(palavras_validas)
    return texto_final, palavras_validas

Writing utils/processamento.py


In [10]:
%%writefile utils/raspagem_bs4.py
import requests
from bs4 import BeautifulSoup
import time

def raspar_com_bs4(lista_urls):
    tempo_inicio = time.time()
    texto_total = ""

    for url in lista_urls:
        resposta = requests.get(url)
        if resposta.status_code == 200:
            sopa = BeautifulSoup(resposta.content, 'html.parser')
            paragrafos = sopa.find_all('p')
            for p in paragrafos:
                texto_total = texto_total + p.get_text() + " "

    tempo_fim = time.time()
    return texto_total, tempo_fim - tempo_inicio

Writing utils/raspagem_bs4.py


In [11]:
%%writefile utils/raspagem_scrapy.py
import scrapy
from scrapy.crawler import CrawlerRunner
from crochet import wait_for
import time

class SpiderWikipedia(scrapy.Spider):
    name = "wiki_spider"

    def __init__(self, start_urls, lista_textos, *args, **kwargs):
        super(SpiderWikipedia, self).__init__(*args, **kwargs)
        self.start_urls = start_urls
        self.lista_textos = lista_textos

    def parse(self, response):
        paragrafos = response.css('p::text').getall()
        texto_junto = " ".join(paragrafos)
        self.lista_textos.append(texto_junto)

@wait_for(timeout=60.0)
def rodar_spider(urls, lista_textos):
    runner = CrawlerRunner()
    return runner.crawl(SpiderWikipedia, start_urls=urls, lista_textos=lista_textos)

def raspar_com_scrapy(lista_urls):
    tempo_inicio = time.time()
    lista_de_textos = []
    rodar_spider(lista_urls, lista_de_textos)
    texto_total = " ".join(lista_de_textos)
    tempo_fim = time.time()
    return texto_total, tempo_fim - tempo_inicio

Writing utils/raspagem_scrapy.py


In [12]:
%%writefile app.py
import streamlit as st
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from crochet import setup

# Importando as nossas funções da pasta utils
from utils.processamento import limpar_texto
from utils.raspagem_bs4 import raspar_com_bs4
from utils.raspagem_scrapy import raspar_com_scrapy

# Inicia o crochet
setup()

def criar_url_wiki(termo):
    termo_formatado = termo.strip().replace(" ", "_")
    return "https://pt.wikipedia.org/wiki/" + termo_formatado

st.title("Trabalho de Web Scraping: Wikipedia")

metodo_escolhido = st.radio("Escolha a biblioteca para extrair os dados:", ("Requests + BeautifulSoup", "Scrapy"))

termos_padrao = "Universidade Federal do Rio Grande do Norte, Ciência de Dados, Aprendizado de Máquina, Engenharia de Software, Armazém de Dados"
entrada_termos = st.text_area("Digite 5 termos separados por vírgula:", termos_padrao)
palavra_busca = st.text_input("Digite uma palavra para ver quantas vezes ela aparece (ex: software):")

if st.button("Iniciar Scraping"):
    lista_termos = entrada_termos.split(",")
    urls_para_raspar = []
    for termo in lista_termos:
        urls_para_raspar.append(criar_url_wiki(termo))

    st.write("Extraindo textos da Wikipedia, aguarde...")

    if metodo_escolhido == "Requests + BeautifulSoup":
        texto_bruto, tempo = raspar_com_bs4(urls_para_raspar)
    else:
        texto_bruto, tempo = raspar_com_scrapy(urls_para_raspar)

    st.success(f"Tempo demorado com {metodo_escolhido}: {tempo:.2f} segundos")

    if texto_bruto != "":
        texto_limpo, lista_palavras = limpar_texto(texto_bruto)

        st.subheader("Nuvem de Palavras")
        nuvem = WordCloud(width=800, height=400, background_color='white').generate(texto_limpo)
        figura, eixo = plt.subplots()
        eixo.imshow(nuvem, interpolation='bilinear')
        eixo.axis("off")
        st.pyplot(figura)

        st.subheader("Contagem de Palavra Específica")
        if palavra_busca != "":
            palavra_formatada = palavra_busca.lower().strip()
            quantidade = lista_palavras.count(palavra_formatada)
            st.write(f"A palavra **{palavra_formatada}** apareceu **{quantidade}** vezes no texto inteiro.")
    else:
        st.error("Nenhum texto foi encontrado.")

Overwriting app.py


In [ ]:
!wget -q -O - ipv4.icanhazip.com
!streamlit run app.py & npx localtunnel --port 8501